In [1]:
### Code for taking the estimates from our plotter equations

### Import Library

import numpy as np # Generalized mathematical operations
import matplotlib.pyplot as plt# Generalized plotting
%matplotlib inline
import pandas as pd # For manipulating dataframes
import os as os #For manipulating directory values
from lmfit import Parameters, minimize
from scipy.interpolate import CubicSpline # For the comparison function between my simulation and my experiment


In [2]:
### Simulation parameters, or numbers I need to define to run the simulation:

#Define my pixel-micron conversion ratio
starter_microns = 1569.2971373332712 # We want our simulation to span this number of microns. 
starter = 75 # number of spaces desired in list, can vary.
starter_ratio = starter_microns/starter # Conversion ratio for distance per index, or (microns/index)

# Get the value of the gel width in units of pixels
#alpha = find_alpha(starter_ratio,j) # gel_width, with input of ratio for distance per index
length = (starter-4) # This code might be defunct

# Fits from Mathematica MolarConcentration_Full_NADH_Calibration.nb document. These fits convert intensity measurements to 
# NADH concentration
Fit1 = -0.12123625148063022
Fit2 = 0.00046298638444968403 # Gives NADH concentration in mM, in the formulation NADH = Fit1 + Fit2*MicroscopeIntensity

#Discretize time and space

T = 1800.0  # total time, seconds
dt = 0.1  # time step, in seconds
n = int(T / dt)  # number of iterations
dx = starter_ratio #1 # Distance between index points

### NADH consumption physical and chemical parameters:

# Enzyme activity rates, in seconds:
kLDH = 0.9 # milliMolar per second
kPK = 0.6 # milliMolar per second
#kmy = 1.0 #milliMolar per minute or micromoles per minute per mL

# Label if you want to plot your enzyme activity rate constant
kLDH_label = "0,6"


# Diffusion Constant
D = 1000.0/(starter_ratio**2) # Gives Diffusion as microns^2/s to spaces^2/s. Approx. 17.75 indexes^2/s. Defunct now.

gamma = -0.00016268876356871281   #Bleaching Constant, calculated from program: "Kymograph_BleachingEstimate"
# Need to check and specify that gamma is in milliMolar per seconds

In [3]:
# Saved results for myosin activity rate found by running Nelder mead search algorithm on previous experiments. (mM/s)
kmy_result = [0.0013165274170419128, 0.001491050522094639, 0.001362683705508072, 0.004087037028677409, 0.006837348404687527, 
      0.002244136067481195, 0.0014282037724002716, 0.010320089042819447, 0.013701627679458817, 0.016491178626356184, 
      0.0013430940923084833, 0.00011723981525602056]

# Saved results for diffusion found by running Nelder mead search algorithm on previous experiments. (microns^2/s)
D_result = [95.48825528366228, 137.38858248492636, 161.26301525526256, 84.299068008207, 81.70746657749527, 206.07926426555423, 
     77.10989720963045, 134.19552029608442, 
     66.83111355341607, 223.58249258644562, 58.543961145226326, 139.18696870313545]

In [4]:
### Define function locations
# This code sets up some of the definitions we need to run these programs

# Change directory to master directory to make ease of use possible.
os.chdir('C:/Users/franc/Box/ALAB Data Francis Cavanna/2021 Actomyosin Control/')

# Set the name of the master directory
base = 'C:/Users/franc/Box/ALAB Data Francis Cavanna/2021 Actomyosin Control/'

# List of folders containing data you would want to process
List_folder = ['2022_1_28', '2022_1_31', '2022_2_25','2022_3_13_Anthony/3-13-2022/0,6_F-0,12_M-1', 
               '2022_3_13_Anthony/3-13-2022/0,6_F-0,12_M-2','2022_3_13_Anthony/3-13-2022/0,6_F-0,12_M-3',
              '2022_3_13_Anthony/3-13-2022/1,2 F-0,06 M-1', '2022_3_13_Anthony/3-13-2022/1,2 F-0,12 M-1',
               '2022_3_15_Anthony/1,2 F-0,24 M-1','2022_3_15_Anthony/1,2 F-0,24 M-2','2022_4_16','2023_2_03/fascin']

# Standard name of file in each folder.
Base_name = '/Results/MattiaRep.csv'


# List of csv files containing the estimated perimeters for the experiment in question and
# the approximated areas of these experiment results. May or may not be useful.
Files = ['RF=1_10  RM=1_50_NADH_Time0.csv','RF=1_10  RM=1_200_NADH_Time1.csv','RF=1_10  RM=1_100_NADH_Time2.csv',
        'RF=1_20  RM=1_100_NADH_Time3.csv','RF=1_20  RM=1_100_NADH_Time4.csv','RF=1_20  RM=1_100_NADH_Time5.csv',
        'RF=1_10  RM=1_200_NADH_Time6.csv','RF=1_10  RM=1_100_NADH_Time7.csv','RF=1_10  RM=1_50_NADH_Time8.csv',
        'RF=1_10  RM=1_50_NADH_Time9.csv','RF=1_10  RM=1_100_NADH_Time10.csv','RF=1_10  RM=1_200_NADH_Time11.csv']

In [5]:
### Function to return size of actomyosin gel in pixels for all time
def find_alpha(ratio,j): #Takes an argument relating microns to pixels
    
    # Import csvs with length scale and write them to a list
    Lengths_list = []

    # Iterate through csvs and append to list
    for i in range(len(List_folder)):
        # Import the csvs to the program.
        df_length = pd.read_csv(base+ List_folder[i]+Base_name)
        # Append the csvs to a list.
        Lengths_list.append(list(df_length.iloc[0,1:]))
    
    # Average the values of length across all gels:
    #alpha_list = [0 for x in Lengths_list[0]]
    
    
    alpha_list = Lengths_list[j]
    
    # Divide each element by the ratio between pixel and total length to get the list of lengths in microns.
    alpha_list = [x/ratio for x in alpha_list] 
        
    return alpha_list # When selecting from this list by array, you input alpha[seconds*1 index/20 seconds]



In [6]:
# Subtract the noise floor from the dataframe
def noise_correction(experiment):
    # Find width of noise floor
    Eighty = experiment.iloc[:,80]
    Ninty = experiment.iloc[:,90]
    
    Normalized = Ninty.sub(Eighty,axis=0)
    
    # Get list of indices of noise floor
    indices = []
    for i in range(len(Normalized)):
        if (Normalized.iloc[i] >= -0.01):
            indices.append(Eighty.iloc[i])
        else:
            break
    
    # Calculate mean of noise floor, with condition for no values being found
    if len(indices) > 0:
        mean_floor = np.mean(indices)
    else: # If there are no indices where the noise floor is positive
        mean_floor = 0
    
    # Subtract average noise floor from all data.
    Clean_exp = experiment.iloc[:,0:90].sub(mean_floor)
    #Clean_exp = Clean_exp.insert(91, "L (microns)", experiment.iloc[91], True)
    
    # Return the cleaned noise floor
    return Clean_exp

In [7]:
### Calculate stabiility parameter to sense how stable your simulation is. Less than 0.5 necessary, above 0.1 is dangerous.

st = dt*D/starter_ratio**2 
print(str(st)+" Is the stability parameter, which must be less than 0.5") # This isn't actually true! 0.5 is necessary for
                                                                # a numerical approximation for euler methods (I believe)
                                                                # but we don't run Euler methods
print("n (step number) is equal to "+ str(n))

0.0005217041885212224 Is the stability parameter, which must be less than 0.5
n (step number) is equal to 18000


In [8]:
# Define a function for calculating myosin activity

def myoK_2D(ATP,alpha_value,t,kmy,difference):#Pass 2D ATP array to myoK, get updated ADP array with myosin value
    #alpha_value = int(np.ceil(alpha))-2 # Find the region of ADP production, including the border
                                              # We pass the correct value of time in the argument of the function
                                                # We also pass ATP[2:-2] as an argument to the function, rather than 
                                                # correcting for it here
    #difference = starter - alpha_value - 2
    
#     print(difference)
#     print(alpha_value)
    
    L1 = ATP[0:alpha_value,0:alpha_value]*kmy # Take existing location of ATP truncate it to where myosin is
    
    # Now make additional matrices of zero activity and join to matrix defined by alpha_value
    L2 = np.random.randint(1,size=(alpha_value,difference))
    L3 = np.random.randint(1,size=(difference,alpha_value))
    L4 = np.random.randint(1,size=(difference,difference))
    
    # Join the matrices into something sensible
    L5 = np.concatenate((L1,L2),axis=1)
    L6 = np.concatenate((L3,L4),axis=1)
    L7 = np.concatenate((L5,L6),axis=0)
    
    return L7 # Return a complete list representing ADP generated by only and exclusively myosin activity

In [9]:
# Desirable to expand to 2 Dimensions. Bring back the 2D Laplacian!
# Make a function that computes the 2nd derivative of a line in 1D space, using 2D five-point stencil.
def laplacian2D(Z):
    Ztop = Z[0:-2, 1:-1]
    Zleft = Z[1:-1, 0:-2]
    Zbottom = Z[2:, 1:-1]
    Zright = Z[1:-1, 2:]
    Zcenter = Z[1:-1, 1:-1]
    return (Ztop + Zleft + Zbottom + Zright - 4 * Zcenter) / dx**2

In [10]:
def expand_symmetric_array(arr):# Takes ONLY Lists as arguments, returns an array with one indice larger in all dimensions
    if not arr:
        return arr  # Return an empty array if the input is empty

    # Get the dimensions of the original array
    rows, cols = len(arr), len(arr[0])

    # Create a new array with expanded dimensions
    expanded_arr = [[arr[i][j] for j in range(cols)] for i in range(rows)]

    # Expand the rows and columns by one
    rows += 2
    cols += 2

    # Fill the top and bottom borders
    for i in range(rows):
        if i == 0:
            tag = [arr[0][:]]
            for j in range(cols-2):
                tag.append(expanded_arr[j])
            expanded_arr = tag
        elif i == rows - 1:
            #expanded_arr.append(arr[0][:] if i == 0 else arr[-1][:])
            expanded_arr.append(arr[-1][:])

    # Fill the left and right borders
    for i in range(0, rows):
        for j in range(cols):
            if j == 0:
                expanded_arr[i].insert(0, expanded_arr[i][0])
            elif j == cols - 1:
                expanded_arr[i].append(expanded_arr[i][-1])

    
                
    return np.array(expanded_arr).reshape(rows,cols)

In [11]:
# The function evolving the reaction-diffusion system one timestep 
def diff_eq_system(i,kmy,D,L,P,AD,AT,PEP,L0): 
    # Compute the Laplacian of L, applying the 1D function we defined:
    deltaL = laplacian2D(L)
    # Laplacian for Pyruvate
    deltaP = laplacian2D(P)
    # Laplacian for ADP
    deltaA = laplacian2D(AD)
    # Laplacian for ATP
    deltaATP = laplacian2D(AT)
    # Laplacian for PEP
    deltaPEP = laplacian2D(PEP)

    # Call alpha value
    a_i = alpha[int(i*dt/20)]
    
    # Perform computations for ADP and ATP (We'd perform them twice if we did these in the function myoK_2D)
    alpha_value = int(np.ceil(a_i))-2
    difference = starter - alpha_value - 2
    a_0 = alpha[0]
    kmy_consv = (a_0**2/a_i**2)*kmy
    
    # Reaction-Diffusion differential equations here:
    
    L_step = L[1:-1, 1:-1] + dt*(D*1.89* deltaL + gamma*L0 - kLDH* np.multiply(P[1:-1, 1:-1],L[1:-1, 1:-1]))
    P_step = P[1:-1, 1:-1]   + dt*(D*3.16* deltaP - kLDH*np.multiply(P[1:-1, 1:-1],L[1:-1, 1:-1])\
                                    + kPK* np.multiply(PEP[1:-1, 1:-1],AD[1:-1, 1:-1]) )
    AD_step = AD[1:-1, 1:-1] + dt*(D* deltaA + myoK_2D(AT[1:-1, 1:-1],alpha_value,n,kmy_consv,difference) - \
                               kPK* np.multiply(AD[1:-1, 1:-1],PEP[1:-1, 1:-1]) )
    AT_step = AT[1:-1, 1:-1] + dt*(D* deltaATP - myoK_2D(AT[1:-1, 1:-1],alpha_value,n,kmy_consv,difference) + \
                               kPK* np.multiply(AD[1:-1, 1:-1],PEP[1:-1, 1:-1]) )
    PEP_step = PEP[1:-1, 1:-1] + dt*(D*3.16*deltaPEP - kPK* np.multiply(AD[1:-1, 1:-1],PEP[1:-1, 1:-1]))

    # Apply Neumann conditions: derivatives at the edges of the array are 0, so array values at edge take values of center
    #Expand the array and refresh the matrix
    # Convert matrix to list first to get proper functional behavior
    L = expand_symmetric_array(L_step.tolist()).clip(min=0)
    P = expand_symmetric_array(P_step.tolist()).clip(min=0)
    AD = expand_symmetric_array(AD_step.tolist()).clip(min=0)
    AT = expand_symmetric_array(AT_step.tolist()).clip(min=0)
    PEP = expand_symmetric_array(PEP_step.tolist()).clip(min=0)
    
    return L,P,AD,AT,PEP

In [12]:
#Run a full simulation for n steps. Takes kmy, Diffusion, and mean+std NADH
# concentration for both higher and lower bounds as arguments.
# Returns 2D NADH concentration field with corresponding times

def simulation(n,kmy,D,Implict_Lh,Implict_Ll): 
    # Initialize arrays
    L = np.array(np.random.randint(Implict_Ll,high=Implict_Lh,size=(starter,starter),dtype='int64')/1000.0) #NADH field, in mM.
    P = np.array(np.random.randint(1,size=(starter, starter)),dtype='int64')
    AD = np.array(np.random.randint(0,high=1,size=(starter, starter),dtype='int64'))
    AT = np.array(np.random.randint(80,high=110,size=(starter, starter),dtype='int64')/1000)
    PEP = np.array(np.random.randint(160,high=200,size=(starter,starter),dtype='int64')/100.0)
    L0 = np.mean(L)
    
    # Initialize array of NADH concentrations
    Conc_tracker = []
    Time_tracker = []
#     fig, axes = plt.subplots(3, 3, figsize=(12, 12))
#     step_plot = n // 9  # n = 9.0/.001 steps -- 9000 steps!
#                         # Also, double backslash is floor division
        # We simulate the PDE with the finite difference method.
    for i in range(n):
        Plotting = L
        
        # Evolve the simulation by one timestep
        L,P,AD,AT,PEP = diff_eq_system(i,kmy,D,L,P,AD,AT,PEP,L0)# time, myosin activity rate, 
                                                                        # diffusion ATP, diffusion ADP
        if ((i+1)*dt)%360.0 == 0.0:# Target 5 steps to review
            #print(i*dt)
            Conc_tracker.append(L)
            Time_tracker.append(i)
            #print(i*dt/20.0)
#         if i*dt == 180.0:
#             Conc_tracker.append(L)
#             Time_tracker.append(i)
#         elif i*dt == 1799.0:
#             Conc_tracker.append(L)
#             Time_tracker.append(i)
#         if n%8999 == 0:
#             print("Iteration!")
            
    return Conc_tracker,Time_tracker
    
# Modify function that returns an object that take inputs of time and position and outputs values of concentration. Then 
# lmfit model can accept this output and do stuff with it and minimize. Must verify that odeint solves these values
# sequentially, and that odeint's result is what's used by model in Jake's code.

#sim = simulation(n,1.0,1200/starter_ratio**2)

In [13]:
# Function that selects from 75x75 assay in the x, y, and diagonal planes.
# This function takes an argument of length, an argument of alpha, and a conversion factor.
def radial_Extraction(length,chem):
    if (length*1.414 + np.min(alpha)/2 > starter):
        print("oops! Error!")
        return 0.0
    else:
        #Re-index for x and y plane:
        radial_dist = []
        list_len = []
        for i in range(length):
            list_len.append(round(i*1.414213562))
            radial_dist.append(round(i*1.414213562)*starter_ratio)
    #Compile three measures of the 2D array from the x, y, and diagonal directions
    
    kymogs = [] # list of kymographs corresponding to my inputted NADH arrays
    for k in range(len(chem)):# Take only the number of states represented in my arrays of NADH.
        TwoD_field = chem[k]
        chem_Mst = [] #The current kymograph across the range of list_len
        #Extract three lists from chem[k] in the x, y, and diagonal direction, using np.min(alpha) as the starting point.
        for i in range(length):
            chem_Mst.append(np.mean([TwoD_field[round(np.min(alpha)/2),(round(np.min(alpha)/2)+list_len[i])],\
                                     TwoD_field[(round(np.min(alpha)/2)+list_len[i]),round(np.min(alpha)/2)],\
                                    TwoD_field[round(np.min(alpha)/2)+i,round(np.min(alpha)/2)+i]]))
        kymogs.append(chem_Mst)

    # returns kymogs, a list of lists featuring the kymographs associated with the list of lists in chem.
    return kymogs,radial_dist


In [14]:
def find_intr(arr,dist):# Function to interpolate my simulation in preparation for direct comparison with experimental data.
    spl = CubicSpline(dist,arr)
    return spl

def exp_compare(params,j,exp,Implict_Lh,Implict_Ll,df):
    # Function that returns a goodness-of-fit measure between my experiment and simulation for a given series of steps n,
    # with given parameters of diffusion and myosin activity rate.

    kmy = params['kmy'].value
    D = params['D'].value
    length = 43 # a length scale that goes from the center of the simulated actin gel to the edges
    
    # Run the simulation 
    chem = simulation(n,kmy,D,Implict_Lh,Implict_Ll)
    # extract kmyographs from simulation
    chem_Mst,radial_dist = radial_Extraction(length,chem[0])
    
    # Interpolate across the simulation kmyographs
    sim = []
    for i in range(len(chem_Mst)):
        #print(kmy,D)
        spline = find_intr(chem_Mst[i],radial_dist)
        #select NADH values at appropriate micron measurements using list of micron measurements from experimental kymographs
        sim_comp = spline(NADH_Data.loc[:,'L (microns)'])
        sim.append(sim_comp)

    # Get the microns measurements from experimental data.
    range_variable = NADH_Data.loc[NADH_Data.loc[:,'L (microns)'] > np.max(radial_dist)].index.tolist()[0]-1 
    # Calls from df, defined outside

    # Get error for both points in time
    error = find_error(sim,NADH_Data.iloc[:,[18,36,54,72,90]],range_variable)
    
    return error# Returns fit comparing final simulation state to microscope data.

# Collect error for all times in experiment. Error is difference between experiment values and simulation values.
def find_error(sim, exp,range_variable):
    error_grid = []
    for i in range(len(sim)):
        error = [(sim[i][j]-exp.iloc[j,i]) for j in range(range_variable)] # .iloc's indexing is reversed relative to 
                                                                        #list comprehension
        error_grid.append(error)
    flat_error = [i for row in error_grid for i in row]
    return flat_error

In [15]:
# Subtract the noise floor from the dataframe
def noise_correction(experiment):
    # Find width of noise floor
    Eighty = experiment.iloc[:,80]
    Ninty = experiment.iloc[:,90]
    
    Normalized = Ninty.sub(Eighty,axis=0)
    
    # Get list of indices of noise floor
    indices = []
    for i in range(len(Normalized)):
        if (Normalized.iloc[i] >= -0.01):
            indices.append(Eighty.iloc[i])
        else:
            break
    
    # Calculate mean of noise floor, with condition for no values being found
    if len(indices) > 0:
        mean_floor = np.mean(indices)
    else: # If there are no indices where the noise floor is positive
        mean_floor = 0
    
    # Subtract average noise floor from all data.
    Clean_exp = experiment.iloc[:,0:90].sub(mean_floor)
    #Clean_exp = Clean_exp.insert(91, "L (microns)", experiment.iloc[91], True)
    
    # Return the cleaned noise floor
    return Clean_exp

In [17]:
# Plot the simulation vs the data

def plot_color(kmy,D,file,n,j,Implict_Lh,Implict_Ll):
    
    # Select times to plot simulation vs data for graphing
    points = [18,36,54,72,90]
    
    length = 43 
    #print(length*1.414 + np.min(alpha)/2) # maximum length is the x-axis, or starter variable, which is 75 right now.
    filepath = base + 'ProcessingStrainRate_Files//Results//NADH_Micron//'+ file
    
    # Import Data
    NADH_Data = pd.read_csv(filepath,index_col=False,skiprows=0)
    NADH_Mean = np.mean(NADH_Data.iloc[:,0])
    NADH_Std = np.std(NADH_Data.iloc[:,0])
        
    # Run the simulation for given n, kmy, D, and Implicit bounds of NADH
    chem = simulation(n,kmy,D,Implict_Lh,Implict_Ll) 

    # Get kymograph of NADH concentration from simulation
    chem_Mst,radial_dist = radial_Extraction(length,chem[0]) 
    
    # Select maximum length we can plot from the simulation.
    range_variable = NADH_Data.loc[NADH_Data.loc[:,'L (microns)'] > np.max(radial_dist)].index.tolist()[0]-1
    
    # Plot the figure of simulation and experiment data
    fig = plt.figure(figsize=(8,6))
    ax = fig.add_subplot(111)
    
    ax.set_ylim([0.0, 1.2])
    
    # Plot multiple points in the figure
    for i in range(len(points)):
        spline = find_intr(chem_Mst[i],radial_dist)
        # Data structure of saved NADH concentration is as follows: Concentration [0,9], Location [10]

        microns = NADH_Data.loc[:,'L (microns)'] #select my micron ranges to return list of distances in microns
        sim_comp = spline(NADH_Data.loc[:,'L (microns)']) #select my micron ranges to return list of distances in microns
    
        index_select = points[i]
        # Select index for data
        clean_exp = noise_correction(NADH_Data)
        if (index_select == 90):
            exp_comp = clean_exp.iloc[:,89]
        else:
            exp_comp = clean_exp.iloc[:,index_select]

        # Plot the simulation vs data at different times.
        if (i==0):
            ax.scatter(microns[0:range_variable],sim_comp[0:range_variable],s=45, c = 'black',label="6.0 min")
            ax.scatter(microns[0:range_variable:10],exp_comp[0:range_variable:10],s=20, c = 'black')
        elif (i==1):
            ax.scatter(microns[0:range_variable],sim_comp[0:range_variable],s=45, c = 'blue',label="12.0 min")
            ax.scatter(microns[0:range_variable:10],exp_comp[0:range_variable:10],s=20, c = 'blue')
        elif (i==2):
            ax.scatter(microns[0:range_variable],sim_comp[0:range_variable],s=45, c = 'purple',label="18.0 min")
            ax.scatter(microns[0:range_variable:10],exp_comp[0:range_variable:10],s=20, c = 'purple')
        elif (i==3):
            ax.scatter(microns[0:range_variable],sim_comp[0:range_variable],s=45, c = 'red',label="24.0 min")
            ax.scatter(microns[0:range_variable:10],exp_comp[0:range_variable:10],s=20, c = 'red')
        elif (i==4):
            ax.scatter(microns[0:range_variable],sim_comp[0:range_variable],s=45, c = 'orange',label="30.0 min")
            ax.scatter(microns[0:range_variable:10],exp_comp[0:range_variable:10],s=20, c = 'orange')
            
    plt.xlabel('radial distance $r \: (\mathrm{\mu m})$',fontsize=20)
    plt.ylabel('[NADH] (mM)',fontsize=20,rotation=0)
    ax.yaxis.set_label_coords(-0.1,1.02)
    
    plt.tick_params(axis='both', which='major', labelsize=16)
    
    # Put a legend to the right of the current axis
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    
    plt.title('Simulation compared to experimental data')

    plt.savefig(base + "ProcessingStrainRate_Files\\Results\\Sim_Exp_Comparison_Exp"+str(j)+".png",dpi="figure",format="png",bbox_inches='tight')

In [18]:
# These three cells are breaker lines dividing between the program and functions.

In [19]:
# Define the lmfit Model for the differential equations
#model = Model(simulation)

# Set initial guesses for the parameters 
params = Parameters()
params.add('kmy', value=0.003383162004690288, min=0.0)
params.add('D', value = 100.0, min = 0.0)

In [ ]:
#Run the simulation code and plot the NADH concentration from estimated diffusion and myosin activity coefficients

for j in range(0,12):
#for j in [0]:
    kmy = kmy_result[j]# Plot the results of my data
    Diff = D_result[j]
    
    filepath = base +\
    'ProcessingStrainRate_Files\\Results\\NADH_Micron\\'+Files[j]
    NADH_Data = pd.read_csv(filepath,index_col=False,skiprows=0)
    
    #Clean the experiment data
    clean_exp = noise_correction(NADH_Data)
    exp_microns = NADH_Data.iloc[91]

    # Define upper and lower bounds for starting NADH concenration
    Implict_Lh = int(1000*(np.mean(clean_exp.iloc[:,0]) + np.std(clean_exp.iloc[:,0])))
    Implict_Ll = int(1000*(np.mean(clean_exp.iloc[:,0]) - np.std(clean_exp.iloc[:,0])))
    
    alpha = find_alpha(starter_ratio,j) # gel_width, with input of ratio for distance per index
    
    # make plots for the data at different times.
    plot_color(kmy,Diff,Files[j],n,j,Implict_Lh,Implict_Ll)



In [ ]:
## Plots for NADH concentration vs radial distance in experiment only.

# Get error list as arrays
Error_List = []
radial_dist_Lists = []

for i in range(0,12):
#     fig = plt.figure(figsize=(8,6))
#     ax1 = fig.add_subplot(111)

#     # Plot individual curves at different times.
#     for k in range(len(SampleCurves)):
#          ax1.plot(microns_Val, SampleCurves[k],label="t = "+str(k*200)+"$\it{s}$") # 

#     # Set label for y-axis and adjust position
#     plt.ylabel('[NADH] (mM)',fontsize=ftsz,rotation=0)
#     ax1.yaxis.set_label_coords(-0.1,1.02)

#     # Set label for x-axis and adjust position
#     plt.xlabel('$distance \: (\mathrm{\mu m})$',fontsize=ftsz)

#     #Create legend for time parameter
#     plt.legend(bbox_to_anchor=(1.32, 1),loc="upper right",prop={'size': 15})

#     # Create title to save conditions of experiment:
#     plt.title(List_Sheets[i])
    
#     # Enlarge tick parameters
#     plt.tick_params(axis='both', which='major', labelsize=tcksz)
    
#     # Shade out grey region for supplementary information. Comment out as needed.
#     plt.axvspan(microns_Val[-50], microns_Val[-1], color='gray', alpha=0.2)

#     # Save the figure
#     plt.savefig('C://Users//franc//Box//ALAB Data Francis Cavanna//2021 Actomyosin Control//' 
#                 + "ProcessingStrainRate_Files\\Results\\NADH_Micron\\" +List_Sheets[i]+'_NADH_vs_Distance_Greyed'+
#                 str(i)+'.png',dpi="figure",format="png",bbox_inches='tight')
    
    
    
    ######################################################################################################################
    
    # This block of code computes error for the NADH experiment vs simulation
    
    # Run the simulation for the given index.
    
    # Set initial guesses for the parameters.
    kmy = kmy_result[i]
    Diff = D_result[i]
    
    # Add the parameters for the simulation
    params = Parameters()
    params.add('kmy', value=kmy, min=0.0)
    params.add('D', value = Diff, min = 0.0)

    length = 43 # a length scale that goes from the center of the simulated actin gel to the edges

    alpha = find_alpha(starter_ratio,8) # gel_width, with input of ratio for distance per index

    filepath = base + 'ProcessingStrainRate_Files\\Results\\NADH_Micron\\'+Files[i]
    NADH_Data = pd.read_csv(filepath,index_col=False,skiprows=0)

    # Define upper and lower bounds for starting NADH concenration
    Implict_Lh = int(1000*(np.mean(NADH_Data.iloc[:,0]) + np.std(NADH_Data.iloc[:,0])))
    Implict_Ll = int(1000*(np.mean(NADH_Data.iloc[:,0]) - np.std(NADH_Data.iloc[:,0])))
    
    # Run the simulation
    chem = simulation(n,kmy,D,Implict_Lh,Implict_Ll)
    # extract kmyographs from simulation
    chem_Mst,radial_dist = radial_Extraction(length,chem[0])
    radial_dist_Lists.append(radial_dist)

    # Interpolate across the simulation kmyographs
    sim = []
    for i in range(len(chem_Mst)):
        spline = find_intr(chem_Mst[i],radial_dist)
        
        #select NADH values at appropriate micron measurements using list of micron measurements from experimental kymographs
        sim_comp = spline(NADH_Data.loc[:,'L (microns)'])
        sim.append(sim_comp)
    
    # Get the microns measurements from experimental data.
    range_variable = NADH_Data.loc[NADH_Data.loc[:,'L (microns)'] > np.max(radial_dist)].index.tolist()[0]-1 # Calls from df, defined outside

    # Select experiment values for the apropriate 200s point in time. find_error collects difference in experiment values 
    # and simulation values.
    error = find_error(sim,NADH_Data.iloc[:,[18,36,54,72,90]],range_variable)
    Error_List.append(error)

In [ ]:
print(len(Error_List[0]))

In [ ]:
# Plot the error for all datasets on a single curve.

fig = plt.figure(figsize=(8,6))
ax1 = fig.add_subplot(111)

for k in range(0,12):
    filepath = base + 'ProcessingStrainRate_Files\\Results\\NADH_Micron\\'+Files[i]
    NADH_Data = pd.read_csv(filepath,index_col=False,skiprows=0)
    microns = NADH_Data.loc[:,'L (microns)'] #select my micron ranges to return list of distances in microns

    print(len(Error_List[k][0:range_variable]))
    print(np.max(radial_dist_Lists[k]))
    print(k)

    # Get the microns measurements from experimental data.
    range_variable = NADH_Data.loc[NADH_Data.loc[:,'L (microns)'] > np.max(radial_dist_Lists[k])].index.tolist()[0]-1 # Calls from df, defined outside
    microns = NADH_Data.loc[:,'L (microns)'] #select my micron ranges to return list of distances in microns
    
    ax1.plot(microns[0:range_variable],Error_List[k][0:range_variable])
    
# Set label for y-axis and adjust position
plt.ylabel('error (mM)',fontsize=20,rotation=0)
ax1.yaxis.set_label_coords(-0.1,1.02)

# Set label for x-axis and adjust position
plt.xlabel('radial distance $\it{r} \: (\mathrm{\mu m})$',fontsize=20)

plt.title('Simulation error compared to experimental data')

plt.savefig(base + "ProcessingStrainRate_Files\\Results\\Sim_Exp_Error.png",dpi="figure",format="png",bbox_inches='tight')

plt.show()